In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import featuregraph as fg
import numpy as np

print("Python:", sys.executable)
print("FeatureGraph:", fg.__file__)

: 

In [ ]:
bidmc = fg.datasets.bidmc(
    subject=1,
)

: 

In [ ]:
bidmc = fg.oscillation.waves.add_wave_primitives(bidmc, ['respiration'])
bidmc = fg.oscillation.waves.add_wave_id(bidmc, ['respiration'], 'subject')
bidmc = fg.oscillation.waves.add_wave_features(bidmc, ['respiration'], 'respiration_wave_id')

: 

In [ ]:
bidmc['respiration_wave_id']

In [ ]:
from featuregraph.utils.plot import plot

In [ ]:
bidmc.columns.tolist()

In [ ]:
# baseline = (
#     bidmc.groupby('respiration_wave_id')['respiration']
#          .transform('min')
# )

# bidmc['respiration_above_baseline'] = (
#     bidmc['respiration'] - baseline
# )

# bidmc['auc_above_baseline'] = (
#     bidmc.groupby('respiration_wave_id')
#          ['respiration_above_baseline']
#          .cumsum()
# )

bidmc = fg.accumulation.accumulations.add_accumulation_signal(
    bidmc,
    signals=["respiration"],
    thresholds={
        "respiration": "group_min",
    },
    group="respiration_wave_id",
)

bidmc['auc_at_peak'] = np.where(bidmc['exit_respiration_rising'] == 1, bidmc['auc_above_baseline'], 0)
bidmc['is_auc_above_baseline'] = bidmc['auc_above_baseline'] > 0
bidmc['before_peak'] = bidmc.groupby('respiration_wave_id')['exit_respiration_rising'].cumsum() == 0
bidmc['after_peak'] = bidmc.groupby('respiration_wave_id')['exit_respiration_rising'].cumsum() > 0
bidmc['accumulation_before_peak'] = np.where(bidmc['before_peak'] == 1, bidmc['respiration_above_baseline'], 0)
bidmc['accumulation_from_peak'] = np.where(bidmc['after_peak'] == 1, bidmc['respiration_above_baseline'], 0)
bidmc['wave_index'] = bidmc.groupby('respiration_wave_id').transform('cumcount')
bidmc['weighted_acc'] = bidmc['wave_index'] * bidmc['respiration_above_baseline']
bidmc['total_auc'] = bidmc.groupby('respiration_wave_id')['respiration_above_baseline'].transform('sum')
bidmc['half_auc'] = bidmc['total_auc'] * .5
half_acc_time = (
    bidmc.loc[bidmc['reached_half_acc']]
    .groupby('respiration_wave_id')['wave_index']
    .first()
)

bidmc['half_acc_time'] = (
    bidmc['respiration_wave_id'].map(half_acc_time)
)

In [ ]:
bidmc['diff'] = bidmc['respiration'].diff()
fig, axes = plot(bidmc[:2000],[
                               ['half_acc_time']
])

In [ ]:
summarydf = bidmc.groupby('respiration_wave_id').agg(
    total_auc=('auc_above_baseline', 'max'),
    accumulation_before_peak=('accumulation_before_peak', 'sum'),
    accumulation_from_peak=('accumulation_from_peak', 'sum'),
    auc_at_peak=('auc_at_peak', 'max'),
    time_to_max_auc=('is_auc_above_baseline', 'sum'),
    first_moment=('weighted_acc', 'sum'),
    half_acc_time=('half_acc_time', 'first')
    )

In [ ]:
summarydf['accumulation_rate'] = summarydf['total_auc'] / summarydf['time_to_max_auc']
summarydf['accumulation_symmetry'] = 1 - ((summarydf['accumulation_before_peak'] - summarydf['accumulation_from_peak']).abs() / \
                                     (summarydf['accumulation_before_peak'] + summarydf['accumulation_from_peak']))
summarydf['centroid_time'] = summarydf['first_moment'] / summarydf['total_auc']
summarydf.reset_index()

<!-- Possible feature	Meaning
total_auc	Total accumulated quantity
accumulation_rate	Average accumulation per unit time
peak_accumulation_rate	Maximum instantaneous contribution
accumulation_before_peak	Quantity accumulated before peak
accumulation_after_peak	Quantity accumulated after peak
accumulation_symmetry	Fraction accumulated before vs. after peak
normalized_auc	AUC normalized by duration or amplitude
accumulation_efficiency	AUC relative to theoretical maximum
centroid_time	Center of mass of accumulated quantity
half_accumulation_time	Time when 50% of total accumulation has occurred -->

In [ ]:
# Possible feature	Meaning
# total_auc	Total accumulated quantity
# accumulation_rate	Average accumulation per unit time
# peak_accumulation_rate	Maximum instantaneous contribution
# accumulation_before_peak	Quantity accumulated before peak
# accumulation_after_peak	Quantity accumulated after peak
# accumulation_symmetry	Fraction accumulated before vs. after peak
# normalized_auc	AUC normalized by duration or amplitude
# accumulation_efficiency	AUC relative to theoretical maximum
# centroid_time	Center of mass of accumulated quantity
# half_accumulation_time	Time when 50% of total accumulation has occurred